# 03 · Modeling

Every guess and model is tested season by season with `src/evaluation.py`: for each of the last 9 seasons (2000–2009, no 2004), it may only learn from the seasons before, then predicts that season. The score is the average over the 9 seasons.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src.evaluation import test_seasons, evaluate_by_season, division_rank, mean_rank_error

team_table = pd.read_csv("../data/team_table.csv")
seasons = test_seasons(team_table)
seasons

## Task (a): ranking

Score: on average, how many positions off the predicted rank is (0 = perfect).

### Simple guess: same order as last season

Inside each division, the team with the most points per game last season is predicted 1st, the next 2nd, and so on. New teams go last. This is "same position as last season" when divisions stay the same, and still works when they change (e.g. 1998, when the league went from 4 to 6 divisions).

In [ ]:
def same_order_as_last_season(train, test):
    return division_rank(test, test["last_pts_vs_avg"])

simple_a = evaluate_by_season(team_table, same_order_as_last_season, mean_rank_error, seasons)
print("Average positions off:", round(simple_a["score"].mean(), 2))
simple_a

### For comparison: random order

What a guess with no information scores. Any model has to be clearly better than this, and better than the simple guess above.

In [ ]:
rng = np.random.default_rng(0)

def random_order(train, test):
    return division_rank(test, rng.random(len(test)))

random_a = evaluate_by_season(team_table, random_order, mean_rank_error, seasons)
print("Average positions off:", round(random_a["score"].mean(), 2))

### First models

Each model learns, from the training seasons only, a team's **relative position** in its division (`rank / div_size`, so 1st of 5 and 1st of 7 are comparable). Its prediction is then turned into positions 1, 2, 3… inside each division with `division_rank`.

- **Linear regression, 3 features:** last season's points and the two roster columns.
- **Linear regression, all features.**
- **Random forest, all features:** many decision trees averaged; can pick up non-linear patterns.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

ALL_FEATURES = ["last_pts_vs_avg", "last_gf_vs_avg", "last_ga_vs_avg", "last_made_playoffs", "last_rank",
                "new_team", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg", "div_size"]
FEW_FEATURES = ["last_pts_vs_avg", "roster_pts_vs_avg", "roster_goalie_ga_vs_avg"]

def model_predict(make_model, features):
    """A predict(train, test) function for the test loop: learn relative position, return ranks."""
    def predict(train, test):
        model = make_model().fit(train[features], train["rank"] / train["div_size"])
        return division_rank(test, -model.predict(test[features]))   # lower predicted position = better
    return predict

models = {
    "Linear, 3 features": model_predict(LinearRegression, FEW_FEATURES),
    "Linear, all features": model_predict(LinearRegression, ALL_FEATURES),
    "Random forest, all features": model_predict(
        lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0), ALL_FEATURES),
}
results = {"Random order": random_a, "Simple guess": simple_a}
for name, predict in models.items():
    results[name] = evaluate_by_season(team_table, predict, mean_rank_error, seasons)

summary = pd.DataFrame({name: r.set_index("season")["score"] for name, r in results.items()})
print(summary.mean().round(2).sort_values())
summary.round(2)

### What this shows

TODO (group): compare each model with the simple guess, season by season. Is the difference big enough to matter with only 9 test seasons?

## Task (b): coach change

Same loop: `evaluate_by_season(table, predict, score, seasons)`.

- `predict(train, test)` returns 1 (new coach) or 0 for each row of `test`, learning only from `train`.
- `score(test, predictions)` returns one number, e.g. the share of real coach changes caught (recall).